# Extension B: Topic-Personalized PageRank

Standard PageRank models a *random surfer* who clicks links at random and occasionally **teleports** to a completely random page. **Personalized PageRank (PPR)** changes that teleportation rule: instead of jumping to any random page, the bored surfer always returns to a specific **seed page**. Over many iterations, probability mass concentrates around pages reachable from — and topically close to — that seed.

The update rule at each iteration is:

$$x^{(t+1)} = d \cdot P^T x^{(t)} + \text{dangling correction} + (1-d) \cdot v$$

where:
- $v$ is the **personalization vector**: a probability distribution over seed pages (here, all mass on a single page)
- $d = 0.85$ is the damping factor (probability of following a link vs. teleporting)
- $P^T$ is the transposed column-stochastic transition matrix

We implement and compare **two variants** that differ only in how dangling nodes (pages with no outgoing links) redistribute their trapped mass.

In [1]:
import numpy as np
import scipy.sparse as sp
import time
from graph import load

## Step 1: Load the Graph

We load the pre-built Simple English Wikipedia link graph from the `data/` directory. The graph was produced by `graph/parser.py` (streaming XML parse of the Wikipedia dump) and deduplicated by `graph/filter.py`.

In [2]:
print("Loading graph arrays...")
titles, edges, out_degree, dangling, categories = load()
N = len(titles)

print(f"  Pages    : {N:,}")
print(f"  Edges    : {len(edges):,}")
print(f"  Dangling : {dangling.sum():,}  ({100 * dangling.mean():.1f}% of pages)")

Loading graph arrays...


  Pages    : 293,984
  Edges    : 3,546,139
  Dangling : 3,387  (1.2% of pages)


## Step 2: Build the Sparse Transition Matrix P^T

We build the **transposed** transition matrix $P^T$ once in CSR (Compressed Sparse Row) format and reuse it for both PPR variants.

Each non-zero entry at position $(\text{target}, \text{source})$ holds $1 / \text{out\_degree}(\text{source})$, meaning a surfer at `source` splits their probability equally among all pages they link to. The matrix-vector product $P^T \cdot x$ then redistributes every page's current score to its neighbors in a single BLAS call — much faster than the scatter-add loop used in the baseline implementation.

In [3]:
print("Constructing P^T transition matrix...")
t0 = time.time()

sources = edges[:, 0]
targets = edges[:, 1]
data    = 1.0 / out_degree[sources]

P_T = sp.csr_matrix((data, (targets, sources)), shape=(N, N))

print(f"  Built in {time.time() - t0:.4f}s")
print(f"  Shape     : {P_T.shape}")
print(f"  Non-zeros : {P_T.nnz:,}")

Constructing P^T transition matrix...
  Built in 0.0363s
  Shape     : (293984, 293984)
  Non-zeros : 3,546,139


## Step 3: Personalized PageRank — Two Boundary Conditions

Each iteration mixes three mass flows:

| Flow | Formula | Personalized? |
|------|---------|---------------|
| Link following | $d \cdot P^T x$ | No — structural |
| Dangling nodes | trapped mass redistributed | **This is the toggle** |
| Boredom teleportation | $(1-d) \cdot v$ | Yes — always to seed |

**Variant 1 — Strict PPR** (academic standard, Haveliwala 2002):  
Surfers trapped at dead-end pages jump back to the seed, just like bored surfers do.  
`dangling_jump = d * lost_mass * v`

**Variant 2 — Global Dangling Leak**:  
Surfers trapped at dead-end pages scatter uniformly to *any* page — the same behaviour as standard (non-personalized) PageRank.  
`dangling_jump = d * lost_mass / N`

Both variants use the same boredom teleportation `(1 - d) * v`, which always points to the seed.

In [4]:
def run_ppr(P_T, dangling, N, v, personalize_dangling=True, d=0.85, epsilon=1e-6, max_iter=100):
    """Power-iteration PPR. Returns the converged score vector."""
    x = np.ones(N, dtype=np.float64) / N
    t_start = time.time()

    for iteration in range(1, max_iter + 1):
        link_mass  = P_T.dot(x)
        lost_mass  = np.sum(x[dangling])

        if personalize_dangling:
            dangling_jump = (d * lost_mass) * v        # snap to seed
        else:
            dangling_jump = (d * lost_mass) / N        # scatter globally

        teleport_jump = (1.0 - d) * v
        x_next = (d * link_mass) + dangling_jump + teleport_jump

        error = np.sum(np.abs(x_next - x))
        x = x_next

        if error < epsilon:
            print(f"  Converged in {iteration} iterations ({time.time() - t_start:.4f}s)")
            return x

    print(f"  Max iterations reached ({time.time() - t_start:.4f}s)")
    return x


def get_top_k(scores, titles, k=15):
    top_indices = np.argsort(scores)[-k:][::-1]
    return [(titles[i], scores[i], i) for i in top_indices]


def print_top_k(results, label):
    print(f"\nTop {len(results)} — {label}")
    print(f"{'Rank':<6} {'Page':<40} Score")
    print("-" * 60)
    for rank, (title, score, _) in enumerate(results, 1):
        print(f"{rank:<6} {title:<40} {score:.6f}")

## Build the Personalization Vector

The personalization vector $v$ is a probability distribution over nodes — here a one-hot vector that places all mass on the seed page. This means every random jump (boredom or dangling-node rescue) sends the surfer directly back to **Alan Turing**.

In [5]:
SEED = "Alan Turing"
seed_idx = titles.index(SEED)

v = np.zeros(N, dtype=np.float64)
v[seed_idx] = 1.0

print(f"Seed : '{SEED}'  (node index {seed_idx})")
print(f"v sums to {v.sum():.1f}  (valid probability distribution)")

Seed : 'Alan Turing'  (node index 6)
v sums to 1.0  (valid probability distribution)


## Variant 1: Strict PPR

Every random jump — whether from boredom or from a dead-end page — sends the surfer back to Alan Turing. This is the mathematically clean definition: the personalization vector $v$ governs *every* non-structural jump, keeping the entire probability distribution anchored to the seed's topic.

In [6]:
print("Running Strict PPR (dangling nodes → seed)...")
scores_strict = run_ppr(P_T, dangling, N, v, personalize_dangling=True)

results_strict = get_top_k(scores_strict, titles, k=15)
print_top_k(results_strict, f"Strict PPR  |  seed: {SEED}")

Running Strict PPR (dangling nodes → seed)...


  Converged in 39 iterations (0.1954s)



Top 15 — Strict PPR  |  seed: Alan Turing
Rank   Page                                     Score
------------------------------------------------------------
1      Alan Turing                              0.152692
2      Computer science                         0.009545
3      London                                   0.006861
4      World War II                             0.006083
5      Computer program                         0.005066
6      Cheshire                                 0.004859
7      Medicine                                 0.004846
8      Ireland                                  0.004759
9      Prison                                   0.004727
10     Artificial intelligence                  0.004721
11     Cryptanalysis                            0.004689
12     Mathematician                            0.004646
13     Elizabeth II                             0.004642
14     Petition                                 0.004610
15     University of Cambridge              

## Variant 2: Global Dangling Leak

Only the 15% boredom factor is personalized — it always returns to Alan Turing. Surfers who get trapped at dead-end pages instead scatter uniformly across all pages, exactly as in standard (non-personalized) PageRank. This is a hybrid approach: mostly personalized, but with a global "leak" from dangling nodes.

In [7]:
print("Running Global Dangling Leak PPR (dangling nodes → uniform)...")
scores_leaky = run_ppr(P_T, dangling, N, v, personalize_dangling=False)

results_leaky = get_top_k(scores_leaky, titles, k=15)
print_top_k(results_leaky, f"Global Dangling Leak PPR  |  seed: {SEED}")

Running Global Dangling Leak PPR (dangling nodes → uniform)...


  Converged in 39 iterations (0.2003s)

Top 15 — Global Dangling Leak PPR  |  seed: Alan Turing
Rank   Page                                     Score
------------------------------------------------------------
1      Alan Turing                              0.151339
2      Computer science                         0.009462
3      London                                   0.006810
4      World War II                             0.006038
5      Computer program                         0.005023
6      Cheshire                                 0.004816
7      Medicine                                 0.004807
8      Ireland                                  0.004721
9      Prison                                   0.004687
10     Artificial intelligence                  0.004679
11     Cryptanalysis                            0.004648
12     Mathematician                            0.004606
13     Elizabeth II                             0.004602
14     Petition                                 

## Comparison: Strict PPR vs. Global Dangling Leak

### Quantitative: How different are the two variants?

In [8]:
# Side-by-side ranking table
print(f"{'Rank':<5} {'Strict PPR':<38} {'Global Leak PPR':<38} Same?")
print("-" * 90)
for i, ((t1, s1, _), (t2, s2, _)) in enumerate(zip(results_strict, results_leaky), 1):
    same = "yes" if t1 == t2 else "NO"
    print(f"{i:<5} {t1:<38} {t2:<38} {same}")

# Score-level differences
diff = np.abs(scores_strict - scores_leaky)
print(f"\nMax score difference  : {diff.max():.2e}")
print(f"Mean score difference : {diff.mean():.2e}")
print(f"Relative difference   : {diff.max() / scores_strict.max() * 100:.2f}% of top score")

Rank  Strict PPR                             Global Leak PPR                        Same?
------------------------------------------------------------------------------------------
1     Alan Turing                            Alan Turing                            yes
2     Computer science                       Computer science                       yes
3     London                                 London                                 yes
4     World War II                           World War II                           yes
5     Computer program                       Computer program                       yes
6     Cheshire                               Cheshire                               yes
7     Medicine                               Medicine                               yes
8     Ireland                                Ireland                                yes
9     Prison                                 Prison                                 yes
10    Artificial intelligen

## Qualitative Analysis: Do the Rankings Make Sense?

The top-15 results effectively reconstruct **Alan Turing's biography through the link graph alone**. We can group the pages into three categories:

---

### Group 1 — Expected / Obvious Links
These validate that the algorithm is working correctly:

| Rank | Page | Why it belongs |
|------|------|----------------|
| 1 | Alan Turing | The seed node — expected to dominate |
| 2 | Computer science | Turing is considered the father of CS |
| 5 | Computer program | Core CS concept, directly linked |
| 10 | Artificial intelligence | The Turing Test — foundational contribution |
| 11 | Cryptanalysis | Bletchley Park, breaking Enigma |
| 12 | Mathematician | His formal academic discipline |
| 15 | University of Cambridge | Where he studied and worked |

---

### Group 2 — Surprising Results With Clear Explanations
These pages would not appear in a global top-15, yet PPR surfaces them because of Turing's specific biography:

| Rank | Page | Explanation |
|------|------|-------------|
| 3 | London | His birthplace; also a high-degree hub — both effects reinforce each other |
| 4 | World War II | His Bletchley Park codebreaking work |
| 6 | Cheshire | Wilmslow, Cheshire — where Turing died in 1954 |
| 7 | Medicine | Linked through his court-ordered hormone treatment (chemical castration) |
| 8 | Ireland | Connected through UK history and biographical context |
| 9 | Prison | His 1952 conviction for "gross indecency" |
| 13 | Elizabeth II | Queen Elizabeth II granted Turing a posthumous royal pardon in 2013 |
| 14 | Petition | The public campaign that preceded the royal pardon |

Together, **Prison → Petition → Elizabeth II** trace the full arc of his persecution and posthumous rehabilitation — a narrative that only PPR (not global PageRank) can surface.

---

### Group 3 — What the Boundary Condition Comparison Tells Us

Both variants produce **identical rankings** with a max score difference of less than 0.15%. This is not a coincidence:

- Dangling nodes account for only ~1.15% of all pages (3,387 out of 293,984)
- At convergence, those nodes carry very little accumulated probability mass
- Whether that small mass snaps back to the seed or scatters globally makes a negligible difference to the overall distribution

**Conclusion:** For this graph, the boundary condition choice does not matter in practice. The strict variant is the theoretically correct default (it keeps the full probability mass within the seed's topic), and the leaky variant serves as a useful robustness check confirming the results are stable.

---

### Key Takeaway

Personalized PageRank is not just a ranking algorithm, but also a **topic-aware proximity measure** on the link graph. Seeding at Alan Turing does not simply elevate pages he directly links to. It surfaces the entire contextual neighborhood of his life and work, weighted by how reachable those pages are through the link structure. The rankings read like a biography written by the graph itself.